In [1]:
import pandas as pd
import ast

df_cvs = pd.read_csv("../data/processed/cvs_with_skills.csv")
df_offers = pd.read_csv("../data/processed/job_offers.csv")

# Convert the skills columns back from text into real lists
df_cvs["extracted_skills"] = df_cvs["extracted_skills"].apply(ast.literal_eval)
df_offers["extracted_skills"] = df_offers["extracted_skills"].apply(ast.literal_eval)

print(f"CVs loaded: {len(df_cvs)}")
print(f"Job offers loaded: {len(df_offers)}")

CVs loaded: 224
Job offers loaded: 18


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine all texts (CVs + job offers) so the vectorizer learns one shared vocabulary
all_texts = list(df_cvs["cleaned_text"]) + list(df_offers["description_clean"])

vectorizer = TfidfVectorizer(max_features=500)  # limit to top 500 most informative words
tfidf_matrix = vectorizer.fit_transform(all_texts)

# Split back into CV vectors and job offer vectors
cv_vectors = tfidf_matrix[:len(df_cvs)]
offer_vectors = tfidf_matrix[len(df_cvs):]

print(f"CV vectors shape: {cv_vectors.shape}")
print(f"Job offer vectors shape: {offer_vectors.shape}")

# Compute similarity: every CV compared to every job offer
text_similarity_matrix = cosine_similarity(cv_vectors, offer_vectors)
print(f"\nSimilarity matrix shape: {text_similarity_matrix.shape}")
print("(should be 224 CVs x 18 job offers)")

CV vectors shape: (224, 500)
Job offer vectors shape: (18, 500)

Similarity matrix shape: (224, 18)
(should be 224 CVs x 18 job offers)


In [4]:
def skill_overlap_score(cv_skills, job_skills):
    if len(job_skills) == 0:
        return 0.0
    cv_skills_set = set(cv_skills)
    job_skills_set = set(job_skills)
    overlap = cv_skills_set.intersection(job_skills_set)
    return len(overlap) / len(job_skills_set)  # % of job's required skills that the CV has

# Build a 224 x 18 matrix, same shape as text_similarity_matrix
import numpy as np

skill_matrix = np.zeros((len(df_cvs), len(df_offers)))

for i, cv_skills in enumerate(df_cvs["extracted_skills"]):
    for j, job_skills in enumerate(df_offers["extracted_skills"]):
        skill_matrix[i, j] = skill_overlap_score(cv_skills, job_skills)

print(f"Skill matrix shape: {skill_matrix.shape}")
print(f"Average skill overlap: {skill_matrix.mean():.2f}")
print(f"Max skill overlap: {skill_matrix.max():.2f}")

Skill matrix shape: (224, 18)
Average skill overlap: 0.53
Max skill overlap: 1.00


In [5]:
# Step 5: Calculate experience match score for every CV-job pair (does candidate meet required years?)

def experience_match_score(cv_years, job_years_required):
    if pd.isna(job_years_required):
        return 1.0
    if pd.isna(cv_years):
        return 0.5
    if cv_years >= job_years_required:
        return 1.0
    else:
        return cv_years / job_years_required

experience_matrix = np.zeros((len(df_cvs), len(df_offers)))

for i, cv_years in enumerate(df_cvs["years_experience"]):
    for j, job_years in enumerate(df_offers["years_required"]):
        experience_matrix[i, j] = experience_match_score(cv_years, job_years)

print(f"Experience matrix shape: {experience_matrix.shape}")
print(f"Average experience match: {experience_matrix.mean():.2f}")

Experience matrix shape: (224, 18)
Average experience match: 0.96


In [6]:
# Step 6: Calculate education match score for every CV-job pair (does candidate meet required education level?)

# assign a numeric rank to each education level, so we can compare "higher vs lower"
education_rank = {
    "Not specified": 0,
    "Associate": 1,
    "Bachelor": 2,
    "Master": 3,
    "MBA": 3,  # treated as equal level to Master
    "PhD": 4
}

def education_match_score(cv_education, job_education_required):
    # if job doesn't specify a requirement, don't penalize the candidate
    if job_education_required == "Not specified":
        return 1.0
    cv_rank = education_rank.get(cv_education, 0)
    job_rank = education_rank.get(job_education_required, 0)
    # candidate meets or exceeds the required level
    if cv_rank >= job_rank:
        return 1.0
    # candidate is below the required level - partial credit based on how close they are
    else:
        return cv_rank / job_rank if job_rank > 0 else 0.5

education_matrix = np.zeros((len(df_cvs), len(df_offers)))

for i, cv_edu in enumerate(df_cvs["education"]):
    for j, job_edu in enumerate(df_offers["education_required"]):
        education_matrix[i, j] = education_match_score(cv_edu, job_edu)

print(f"Education matrix shape: {education_matrix.shape}")
print(f"Average education match: {education_matrix.mean():.2f}")

Education matrix shape: (224, 18)
Average education match: 0.87


In [7]:
# Step 7: Combine text similarity, skill overlap, experience match, and education match into one final weighted score

WEIGHT_TEXT = 0.40
WEIGHT_SKILLS = 0.35
WEIGHT_EXPERIENCE = 0.15
WEIGHT_EDUCATION = 0.10

final_score_matrix = (
    WEIGHT_TEXT * text_similarity_matrix +
    WEIGHT_SKILLS * skill_matrix +
    WEIGHT_EXPERIENCE * experience_matrix +
    WEIGHT_EDUCATION * education_matrix
)

print(f"Final score matrix shape: {final_score_matrix.shape}")
print(f"Average final score: {final_score_matrix.mean():.2f}")
print(f"Min score: {final_score_matrix.min():.2f}")
print(f"Max score: {final_score_matrix.max():.2f}")

Final score matrix shape: (224, 18)
Average final score: 0.44
Min score: 0.16
Max score: 0.64


In [8]:
# Step 8: Convert the score matrices into one long results table (CV x job offer x scores)

results = []

for i in range(len(df_cvs)):
    for j in range(len(df_offers)):
        results.append({
            "cv_filename": df_cvs["filename"].iloc[i],
            "job_title": df_offers["job_title"].iloc[j],
            "company": df_offers["company"].iloc[j],
            "text_similarity": round(text_similarity_matrix[i, j], 3),
            "skill_overlap": round(skill_matrix[i, j], 3),
            "experience_match": round(experience_matrix[i, j], 3),
            "education_match": round(education_matrix[i, j], 3),
            "final_score": round(final_score_matrix[i, j], 3)
        })

df_results = pd.DataFrame(results)

print(f"Total CV-job pairs: {len(df_results)}")
df_results.sort_values("final_score", ascending=False).head(10)

Total CV-job pairs: 4032


,cv_filename,job_title,company,text_similarity,skill_overlap,experience_match,education_match,final_score
163,Ami Jape.docx,Senior Project Manager,Razorfish Health,0.104,1.000,1.0,1.0,0.641
117,Akhil_Sr BSA.docx,Sr Business Analyst,"Sunquest Information Systems, Inc.",0.045,1.000,1.0,1.0,0.618
81,AjayKumar.docx,Sr Business Analyst,"Sunquest Information Systems, Inc.",0.040,1.000,1.0,1.0,0.616
3396,Sundar_Java_8+ Years..docx,Senior Java Developer,APPS CONSULTANTS,0.086,0.938,1.0,1.0,0.613
1920,Nikhil.docx,Senior Java Developer,APPS CONSULTANTS,0.083,0.938,1.0,1.0,0.612
1938,Nikith Reddy.docx,Senior Java Developer,APPS CONSULTANTS,0.084,0.938,1.0,1.0,0.612
83,AjayKumar.docx,Salesforce Business Analyst,S&P Global,0.083,0.933,1.0,1.0,0.610
210,Amulya Komatineni.docx,Senior Java Developer,APPS CONSULTANTS,0.081,0.938,1.0,1.0,0.610
624,chenna kesava.docx,Senior Java Developer,APPS CONSULTANTS,0.079,0.938,1.0,1.0,0.610
1128,kalyan das.docx,Senior Java Developer,APPS CONSULTANTS,0.075,0.938,1.0,1.0,0.608


In [9]:
# Step 9: Save the full CV-job matching results to CSV

df_results.to_csv("../data/processed/matching_results.csv", index=False)
print(f"Saved {len(df_results)} CV-job matches to data/processed/matching_results.csv")

Saved 4032 CV-job matches to data/processed/matching_results.csv


In [10]:
# Step 10: Look at the distribution of final scores to choose a sensible compatibility threshold

print(df_results["final_score"].describe())


count    4032.000000
mean        0.435062
std         0.083678
min         0.158000
25%         0.378000
50%         0.441000
75%         0.497000
max         0.641000
Name: final_score, dtype: float64


In [11]:
# Step 11: Create compatibility labels based on threshold, then train Logistic Regression to predict them

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

THRESHOLD = 0.45
df_results["compatible"] = (df_results["final_score"] >= THRESHOLD).astype(int)

print("Label distribution:")
print(df_results["compatible"].value_counts())
print(f"\nCompatible: {(df_results['compatible']==1).sum()} ({(df_results['compatible']==1).mean()*100:.1f}%)")
print(f"Not compatible: {(df_results['compatible']==0).sum()} ({(df_results['compatible']==0).mean()*100:.1f}%)")

X = df_results[["text_similarity", "skill_overlap", "experience_match", "education_match"]]
y = df_results["compatible"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"\nModel accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Label distribution:
compatible
0    2178
1    1854
Name: count, dtype: int64

Compatible: 1854 (46.0%)
Not compatible: 2178 (54.0%)

Model accuracy: 0.97

Classification report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       412
           1       1.00      0.95      0.97       395

    accuracy                           0.97       807
   macro avg       0.97      0.97      0.97       807
weighted avg       0.97      0.97      0.97       807



In [12]:
# Step 12: Apply the trained classifier to all CV-job pairs and add predictions to the results table

df_results["ml_predicted_compatible"] = model.predict(X)
df_results["ml_confidence"] = model.predict_proba(X)[:, 1]  # probability of being "compatible"

print(df_results[["cv_filename", "job_title", "final_score", "compatible", "ml_predicted_compatible", "ml_confidence"]].sort_values("final_score", ascending=False).head(10))

                     cv_filename                    job_title  final_score  \
163                Ami Jape.docx       Senior Project Manager        0.641   
117            Akhil_Sr BSA.docx          Sr Business Analyst        0.618   
81                AjayKumar.docx          Sr Business Analyst        0.616   
3396  Sundar_Java_8+ Years..docx        Senior Java Developer        0.613   
1920                 Nikhil.docx        Senior Java Developer        0.612   
1938           Nikith Reddy.docx        Senior Java Developer        0.612   
83                AjayKumar.docx  Salesforce Business Analyst        0.610   
210       Amulya Komatineni.docx        Senior Java Developer        0.610   
624           chenna kesava.docx        Senior Java Developer        0.610   
1128             kalyan das.docx        Senior Java Developer        0.608   

      compatible  ml_predicted_compatible  ml_confidence  
163            1                        1       0.999876  
117            1       

In [13]:
# Step 13: Save the complete results table (scores + labels + ML predictions) to CSV

df_results.to_csv("../data/processed/matching_results.csv", index=False)
print(f"Saved {len(df_results)} results with classification to data/processed/matching_results.csv")

Saved 4032 results with classification to data/processed/matching_results.csv
